# 📗 [커머스] H&M 데이터 분석 — 01 데이터 이해 & 전처리 (강사용 예시)

**오늘은 강사가 앞에서 풀어 주지 않습니다.** 이 노트북은 프로젝트의 **중심 파트**입니다 — day06~09 에 배운 도구로 **문제를 정의**하고, 데이터의 함정을 눈으로 확인한 뒤, **여러분이 정한 규칙**으로 정제본을 만들어 저장합니다.

> ⚠️ **중요**: 이 노트북 맨 마지막 셀이 `output/hm_clean.csv` 를 저장합니다. **다음 두 노트북(EDA·통계)이 이 파일을 그대로 이어받으므로**, 이 노트북을 끝까지 실행해 저장까지 마쳐야 다음 단계로 넘어갈 수 있습니다.

## 오늘의 파일
| 파일 | 내용 |
|---|---|
| **`01_데이터이해_전처리.ipynb`** | **지금 이 파일** — 문제 정의 + 데이터 이해 + 함정 확인 + 전처리 규칙 수립 + 정제본 저장 |
| `02_데이터_EDA.ipynb` | 이 노트북이 저장한 정제본으로 EDA·시각화 |
| `03_데이터_통계.ipynb` | 통계 분석 + 인사이트 리포트 |

## 이 노트북의 흐름
| 장 | 내용 | 형식 |
|---|---|---|
| 가이드 | day06~08 레시피 요약 — 막힐 때 펼쳐보기 | 참고 |
| 1장 | 문제 정의 — 비즈니스 질문부터 분석 계획까지 | 서술 |
| 2장 | 데이터 이해 — 3테이블 구조 파악 | 제공 코드 + 자가채점 |
| 3장 | 함정 확인 — 이상치·결측 미리보기 | 제공 코드 |
| 4장 | 전처리 규칙 수립 — 결정 기록표 | 서술 |
| 5장 | 테이블 결합 | 제공 코드 + 자가채점 |
| 6장 | 전처리 규칙 적용 & 파생변수 — 처리 로그 | 코드 |
| 7장 | 정제본 저장 | 제공 코드 |

## 세 노트북이 어떻게 이어지는가
이 프로젝트 전체가 **한 문제를 7단계로 꿰는 과정**입니다. 지금 여러분은 **1~3단계**에 있습니다 — 색이 곧 노트북 번호입니다.

![프로젝트 흐름도](../../day10_데이터분석_종합실습/images/프로젝트_흐름도.png)

> 🔧 **강사용 안내**: 이 노트북의 답안·서술 셀에는 **예시 풀이**가 들어 있습니다. **정답이 아니라 "이렇게도 할 수 있다"의 한 사례**입니다. 학생이 문제 정의·결측·이상치 규칙을 다르게 골랐다면 이 예시와 결과가 달라지는 것이 정상입니다.

**자가채점 안내** — 이 노트북은 **원본 3테이블 shape** 와 **조인 직후 행수**, 두 곳만 사실 확인으로 채점합니다. 문제 정의 서술과 결측·이상치 처리 규칙, 최종 정제본의 행수는 **채점하지 않습니다** — 규칙에 따라 숫자가 달라지는 것이 정상입니다.

In [ ]:
# [제공 코드] 이 프로젝트에서 쓸 라이브러리와 한글 폰트를 준비합니다.
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform

pd.set_option('display.max_columns', 40)     # 상품 테이블은 컬럼이 25개라 넉넉히

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(-) 부호 깨짐 방지
sns.set_theme(style='whitegrid', font=KOREAN_FONT, rc={'axes.unicode_minus': False})

## 가이드 — day06~08 레시피 요약
이 프로젝트는 강사 시연이 없습니다. 코드가 기억나지 않으면 아래 표에서 필요한 패턴을 찾아 그대로 복붙해 쓰세요. **이 노트북의 모든 코드 요구는 이 표에서 나옵니다.**

<details><summary>🔧 레시피 표 펼치기 — 막힐 때 여기부터</summary>

| 하고 싶은 것(day06) | 코드 패턴 |
|---|---|
| CSV 읽고 첫인상 보기 | `pd.read_csv(...)` 후 `.head()` / `.info()` / `.describe()` / `.shape` |
| 컬럼 선택 | `df[['a', 'b']]` |
| 컬럼 버리기 | `df.drop(columns=['a', 'b'])` |
| 컬럼 이름 바꾸기 | `df.rename(columns={'old': 'new'})` |
| 라벨/위치로 행 고르기 | `df.loc[조건 또는 라벨]` / `df.iloc[숫자 위치]` |
| 조건 하나로 필터 | `df[df['a'] > 0]` |
| 여러 조건 동시에(AND) | `df[(조건1) & (조건2)]` |
| 목록에 있는 값만 | `df['a'].isin([1, 2])` |
| 범위 안 값만 | `df['a'].between(10, 99)` |
| 정렬해서 극단값 후보 보기 | `df.sort_values('a', ascending=False).head(10)` |
| 결측치 개수 세기 | `df.isna().sum()` |
| 결측치를 값으로 채우기 | `df['a'].fillna(값)` |
| 앞/뒤 값으로 채우기 | `df['a'].ffill()` / `df['a'].bfill()` |
| 자료형 바꾸기 | `df['a'].astype(int)` |
| 문자열 -> 날짜형 | `pd.to_datetime(df['a'])` |
| 문자열에 특정 단어 있는지 | `df['a'].str.contains('단어')` |
| 파생 컬럼 만들기 | `df['new'] = (df['a'] // 10 * 10).astype(int)` |
| 두 테이블 합치고 행 검증 | `m = a.merge(b, on='키', how='inner'); print(len(a), '->', len(m))` |
| 그룹별 집계 | `df.groupby('키')['값'].agg(['sum', 'mean', 'count'])` |
| 컬럼 겹침 확인 | `set(a.columns) & set(b.columns)` |
| **이상치(day08) — IQR 1.5배 규칙** | `Q1, Q3 = df['a'].quantile([.25, .75]); IQR = Q3 - Q1; lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR` |
| **이상치(day08) — Z-score** | `z = (df['a'] - df['a'].mean()) / df['a'].std(); 이상치 = z.abs() > 3` |

</details>

## 1장. 문제 정의
**이 절에서 할 일**: 데이터를 만지기 전에 무엇을 밝힐지 정합니다 — 이후 모든 선택(어떤 컬럼을 볼지, 어떤 그래프를 그릴지)이 여기서 세운 방향을 따라갑니다. 좋은 분석은 좋은 질문에서 시작합니다.

### 여러분에게 맡겨진 상황
여러분은 **H&M 데이터 분석 팀의 데이터 애널리스트**입니다. H&M 은 1947년에 설립돼 전 세계 70여 개국에서 의류·홈·뷰티 상품을 파는 패션 기업이고, 여러분에게 주어진 것은 그 고객·거래·상품 기록입니다. 서비스 현황을 점검해 **고객 경험 개선과 매출 성장 전략**에 쓸 결과물을 만들어야 합니다.

요청받은 것은 네 가지입니다 — 이 프로젝트 세 노트북이 이 네 가지를 나눠 맡습니다.

| # | 요청사항 | 담당 |
| --- | --- | --- |
| 1 | 고객·매출 데이터를 탐색해 **서비스 현황을 시각화**해 주세요 | 02 |
| 2 | **채널·상품군·고객 속성별 매출 특징**을 분석해 주세요 | 02·03 |
| 3 | **이상치·결측치 처리 과정을 포함한 EDA 결과**를 제출해 주세요 | **01(이 노트북)**·02 |
| 4 | **분석 결과와 인사이트(전략)** 를 제공해 주세요 | 03 |

> 이 노트북이 맡은 3번이 나머지의 **바탕**입니다. 여기서 만든 정제본이 잘못되면 02·03 의 그래프와 통계가 전부 잘못된 데이터를 보게 됩니다.

### 1-1. 비즈니스 질문 -> 분석 질문 -> 데이터 질문
막연한 질문은 그대로는 분석할 수 없습니다. 세 단계로 좁혀야 합니다.

1. **비즈니스 질문** — 경영진이 던질 법한 막연한 질문("매출을 어떻게 늘릴까?")
2. **분석 질문** — 측정 가능하게 좁힌 질문("어떤 연령대·채널 조합이 결제금액에 가장 크게 기여하는가?")
3. **데이터 질문** — 정확히 어떤 컬럼·집계로 답할지 못박은 질문("`age_group`·`sales_channel_id` 로 나눠 `price` 합계·평균을 비교하면 어느 조합이 가장 큰가?")

> 위 예시처럼, 3단계를 내려갈수록 **컬럼 이름이 등장할 만큼 구체적**이어야 합니다.

### 1-2. 좋은 질문 / 나쁜 질문

| 나쁜 질문 | 왜 나쁜가 | 좋은 질문으로 고치면 |
| --- | --- | --- |
| "매출이 얼마인가?" | 답이 나와도 결정으로 안 이어짐("그래서 뭘 할 건데?"가 안 생김) | "어느 축(연령·채널·상품군)으로 갈랐을 때 결제금액 차이가 가장 큰가?" |
| "고객들은 어떤가?" | 데이터로 답하기엔 너무 막연함 | "연령대별 평균 결제금액과 구매 빈도는 어떻게 다른가?" |
| "왜 특정 상품이 인기 있나?" | 이 데이터엔 '이유'를 설명할 변수가 없음(관찰 데이터의 한계) | "어떤 상품군이 결제금액 상위인가?"(원인이 아니라 관찰) |

**좋은 질문의 조건**: 데이터에 있는 컬럼으로 답할 수 있다 · 답이 나오면 누군가 결정할 수 있다 · "얼마인가" 보다 "**어느 축으로 갈랐을 때** 차이가 큰가" 가 낫다.

### 1-3. 이해관계자를 정하면 질문이 달라진다
같은 데이터라도 **누구를 위한 분석인지**에 따라 좋은 질문이 달라집니다.

| 이해관계자 | 관심사 | 이 관점의 질문 예시 |
| --- | --- | --- |
| 마케팅팀 | 고객 세그먼트별 프로모션 | 어떤 연령대·멤버십 상태가 프로모션에 반응할 잠재력이 큰가? |
| 상품기획팀 | 인기 상품군·색상 트렌드 | 어떤 제품군·색상이 결제금액 상위를 차지하는가? |
| 경영진 | 채널 구조와 성장 가능성 | 온라인·오프라인 채널별 결제금액 비중과 평균 단가는 어떻게 다른가? |

### 1-4. 가설 쓰는 법 — day09 의 H0/H1 로 연결
가설은 "~일 것이다" 처럼 **데이터로 참·거짓을 가릴 수 있는 문장**이어야 합니다. 이 문장은 나중에 `03_데이터_통계` 에서 배운 통계적 가설검정으로 다시 쓸 수 있습니다 — **귀무가설 H0**(차이가 없다·효과가 없다)과 **대립가설 H1**(차이가 있다)로요.

> 예시: 서술 가설 "연령대가 높을수록 평균 결제금액이 높을 것이다"
> - **H0**: 연령대별 평균 결제금액에 차이가 없다.
> - **H1**: 연령대별 평균 결제금액에 차이가 있다.

지금은 서술만 하면 됩니다 — 실제 검정은 `03_데이터_통계` 의 몫입니다.

**어떤 질문이 어떤 검정으로 가는가** — 아래 네 줄은 `03_데이터_통계` 의 통계 미션 A·B·C·E 와 같은 질문입니다. 지금 세우는 가설이 나중에 어디로 이어지는지 미리 봐 두세요. **질문의 변수 유형(3)이 검정을 결정합니다** — 검정을 먼저 고르는 것이 아닙니다.

![비즈니스 질문을 통계 질문으로 바꾸는 법](../../day10_데이터분석_종합실습/images/비즈니스질문_통계가설_매핑.png)

### 1-5. 분석 계획표 (직접 채우세요)
핵심 질문마다 **필요한 컬럼**과 **볼 방법**을 미리 적어 두면 이후 EDA·통계에서 헤매지 않습니다.

| 질문 | 필요한 컬럼 | 볼 방법(집계·그래프) | 예상 결과 |
| --- | --- | --- | --- |
| 어느 연령대가 결제금액을 주도하는가? | `age_group`, `price` | `groupby(age_group)['price'].agg(['sum','mean'])` + 막대그래프 | 중장년층 평균 단가가 더 높을 것 |
| 온라인·오프라인은 무엇이 다른가? | `sales_channel_id`, `price` | `groupby(channel)['price'].agg(['sum','mean','count'])` | 온라인 건수 비중이 크고 평균단가는 오프라인이 높을 것 |
| 어떤 제품군이 결제금액을 견인하는가? | `product_group_name`, `price` | `groupby(product_group_name)['price'].sum().sort_values()` + 막대그래프 | 상의(Garment Upper body)가 1위일 것 |

### 1-6. 종합 — 비즈니스 목표·이해관계자·핵심 질문·가설 서술
**요구사항** — 1) 이 분석의 **비즈니스 목표를 한 문장**으로 적고("무엇을 위해 이 분석을 하는가"), 2) **이해관계자**를 하나 고르고, 3) **핵심 질문 2~3개**, 4) 각 질문의 **가설**(H0/H1 형태 포함)을 서술하세요.

> 비즈니스 목표는 질문보다 한 단계 위입니다 — 질문은 여러 개지만 목표는 하나이고, 목표가 질문을 고르는 기준이 됩니다. 위 "여러분에게 맡겨진 상황" 의 네 요청사항에서 출발하세요.

**여기까지 되면 통과**: 비즈니스 목표 1문장 + 이해관계자 1개 + 핵심질문 2~3개 + 가설(H0/H1 포함) + 분석 계획표가 채워져 있으면 됩니다. 내용은 사람마다 달라도 됩니다.

**예시 서술**

**비즈니스 목표**: 다음 분기 프로모션 예산을 **어느 고객군·어느 채널에 쓸지** 데이터로 정해 마케팅 효율을 높인다.

**이해관계자**: 마케팅팀 — 다음 분기 프로모션의 타깃 고객군과 채널을 정하는 데 쓴다.

| # | 핵심 질문 | 가설 |
| --- | --- | --- |
| Q1 | 어느 연령대가 결제금액을 주도하는가? | H0: 연령대별 평균 결제금액에 차이가 없다 / H1: 차이가 있다(중장년층이 더 높을 것) |
| Q2 | 온라인·오프라인은 무엇이 다른가? | H0: 채널별 평균 단가에 차이가 없다 / H1: 차이가 있다(오프라인이 더 높을 것) |
| Q3 | 어떤 제품군이 결제금액을 견인하는가? | H0: 제품군별 결제금액 합계에 차이가 없다 / H1: 상의(Garment Upper body)가 가장 클 것 |

## 2장. 데이터 이해 — 3개 테이블
**이 절에서 할 일**: 세 테이블을 각각 불러와 크기·컬럼·자료형·결측을 확인합니다. **분석은 데이터를 눈으로 보는 것에서 시작합니다.** 아래 코드 셀들은 **제공 코드**입니다 — 실행하고 **출력을 읽으세요**. 막히면 가이드의 "CSV 읽고 첫인상 보기" 항목을 참고하세요.

### 데이터 소스 — 출처·구성·관측단위

| 항목 | 내용 |
| --- | --- |
| **출처** | Kaggle 에 공개된 **H&M Personalized Fashion Recommendations** 데이터셋에서 뽑은 표본입니다(H&M 전체 실적이 아닙니다) |
| **구성** | CSV 3개 — 거래 `transactions_hm` · 고객 `customer_hm` · 상품 `articles_hm` |
| **기간** | **2019-01-01 ~ 2019-12-31, 딱 1년치**입니다 |
| **관측단위** | 테이블마다 다릅니다: 거래는 **구매 1건**, 고객은 **고객 1명**, 상품은 **상품 1개**가 한 행입니다. 5장에서 셋을 결합하면 관측단위가 **"고객·상품 정보가 붙은 구매 1건"** 이 됩니다 |
| **주요 변수** | 아래 컬럼 사전 참고 |

> ⚠️ **기간이 1년이라는 점을 기억하세요.** 연도 간 비교나 "몇 년에 걸친 성장 추세" 같은 이야기는 이 데이터로 할 수 없습니다. 월별·요일별 비교까지가 가능한 범위입니다.

### 테이블 관계 (ERD)
![테이블 관계 (ERD)](../../day10_데이터분석_종합실습/images/테이블_관계_ERD.png)
한 고객이 여러 번 구매하고(1:N), 한 상품이 여러 거래에 등장합니다(N:1). **거래 테이블이 중심**이고 양쪽으로 고객·상품 정보를 붙이는 구조입니다. 이 노트북에서는 세 테이블을 각각 `tr`(거래)·`cu`(고객)·`ar`(상품) 으로 부릅니다.

### 컬럼 사전
**거래 `tr` (150,500행)** — 행 = 구매 1건

| 컬럼 | 설명 |
| --- | --- |
| `t_dat` | 구매일(문자열 — 아직 날짜형이 아닙니다) |
| `customer_id` | 고객 식별자 |
| `article_id` | 상품 식별자 |
| `price` | 결제 금액 — **0~1 사이로 정규화된 상대값**(실제 범위 0 ~ 0.422, 평균 0.0276) |
| `sales_channel_id` | 판매 채널(1=오프라인, 2=온라인 — 그 외 값은 3장에서 확인) |

**고객 `cu` (95,516행)** — 행 = 고객 1명

| 컬럼 | 설명 |
| --- | --- |
| `customer_id` | 고객 식별자 |
| `age` | 나이(이상치 섞임 — 3장에서 확인) |
| `club_member_status` | 멤버십 상태(ACTIVE/PRE-CREATE/LEFT CLUB, 일부 결측) |
| `fashion_news_frequency` | 패션 뉴스 수신 빈도 |
| `FN` / `Active` | 뉴스 수신·활성 회원 플래그 |

**상품 `ar` (30,503행, 25컬럼)** — 행 = 상품 1개(주요 컬럼만)

| 컬럼 | 설명 |
| --- | --- |
| `article_id` | 상품 식별자 |
| `prod_name` | 상품명 |
| `product_group_name` | 제품군(Garment Upper body 등) |
| `colour_group_name` | 색상 그룹 |
| `index_group_name` | 타겟 그룹(Ladieswear 등) |

> ⚠️ **읽을 때 주의** — `price` 는 절대 금액이 아니라 정규화값(상대 비교용)입니다. 그리고 이 데이터엔 구매 **수량(quantity) 컬럼이 없어**, `price` 의 합계는 진짜 매출이 아니라 **"구매 단가의 총합"** 입니다. 02·03 에서 "매출"이라는 말을 쓸 때도 이 뜻으로 쓰세요 — 리포트에 그 한계를 한 줄 적어야 합니다.

> ⚠️ **발제문 서술도 데이터로 확인하세요.** 이 프로젝트 발제문에는 `price` 의 통화 단위가 **SEK(스웨덴 크로나)** 라고 적혀 있습니다. 그런데 실제 값을 보면 최대가 0.422 이고 평균이 0.0276 입니다 — 옷 한 벌 값이 0.03 크로나일 수는 없으니, 이 컬럼은 **원 통화 금액이 아니라 정규화된 상대값**입니다. 발제문·기획서에 적힌 설명이 데이터와 어긋나는 일은 실무에서 흔합니다. **주어진 문서를 그대로 믿지 말고 `describe()` 로 직접 확인하는 것**이 분석가의 첫 번째 습관입니다(3장에서 실제로 확인합니다).

In [ ]:
# [제공 코드] 세 테이블 불러오기 (원본 그대로 — 아직 아무것도 손대지 않습니다)
tr = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/transactions_hm.csv')   # 거래
cu = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/customer_hm.csv')       # 고객
ar = pd.read_csv('../../day10_데이터분석_종합실습/data/hm/articles_hm.csv')       # 상품

for name, x in [('transactions', tr), ('customer', cu), ('articles', ar)]:
    # 표가 세로로 맞도록 이름은 14칸 왼쪽, 숫자는 자릿수만큼 오른쪽 정렬한다
    rows, cols = x.shape
    na_count = int(x.isna().sum().sum())
    print(f'  {name:<14} {rows:7d}행 x {cols:2d}열   결측 {na_count:6d}건')

In [ ]:
# [자가채점]
assert tr.shape == (150500, 5)
assert cu.shape == (95516, 6)
assert ar.shape == (30503, 25)
print("✅ 원본 3테이블 shape 확인 통과!")

In [ ]:
# [제공 코드] 거래 테이블 — 이 프로젝트의 중심
print('[앞부분 5행]'); display(tr.head())
print('\n[구조]'); tr.info()
print('\n[수치형 요약]'); display(tr.describe())
print('\n기간:', tr['t_dat'].min(), '~', tr['t_dat'].max())
print('채널 코드별 건수:', tr['sales_channel_id'].value_counts().to_dict())

In [ ]:
# [제공 코드] 고객 테이블
print('[앞부분 5행]'); display(cu.head())
print('\n[구조]'); cu.info()
print('\n나이 요약:'); display(cu['age'].describe())
print('\n멤버십 상태:', cu['club_member_status'].value_counts(dropna=False).to_dict())

In [ ]:
# [제공 코드] 상품 테이블 — 컬럼이 25개라 주요 컬럼만
print('[앞부분 3행]'); display(ar.head(3))
key_cols = ['article_id', 'prod_name', 'product_group_name', 'colour_group_name']
print('\n[주요 컬럼]'); display(ar[key_cols].head())
print('\n제품군 종류:', ar['product_group_name'].nunique(), '개')
display(ar['product_group_name'].value_counts().head(8))

### 컬럼이 많을 때 다루는 법 — `drop`·`rename`·`str.contains` 실습
**왜 하는가**: 상품 테이블은 25컬럼이라 한눈에 보기 어렵습니다. 아래에서 **살펴보기 전용** 축소본을 만들어 봅니다.

> ⚠️ **주의** — 여기서 만드는 `ar_preview` 는 **탐색용 사본**입니다. 실제 정제본(`df`)은 02·03 과의 **스키마 계약(원본 조인 34열)** 때문에 원본 컬럼을 그대로 유지합니다 — `ar` 자체는 바꾸지 않습니다.

In [ ]:
# [제공 코드] drop — _name 짝이 있는 코드/번호 컬럼을 정리한 탐색용 축소본
redundant_cols = ['product_type_no', 'graphical_appearance_no', 'colour_group_code',
                   'perceived_colour_value_id', 'perceived_colour_master_id',
                   'department_no', 'index_code', 'index_group_no', 'section_no',
                   'garment_group_no']
ar_preview = ar.drop(columns=redundant_cols)
print('원본', ar.shape, '-> 축소본', ar_preview.shape)

# rename — 표시용 한글 별칭 (역시 살펴보기 전용, ar 자체는 그대로)
display(ar_preview[key_cols].rename(columns={
    'prod_name': '상품명', 'product_group_name': '제품군', 'colour_group_name': '색상그룹',
}).head())

# str.contains — 상품명으로 탐색하기
is_dress = ar['prod_name'].str.contains('Dress', case=False, na=False)
print("상품명에 'Dress' 가 들어간 상품 수:", int(is_dress.sum()))
display(ar.loc[is_dress, ['prod_name', 'product_group_name']].head(3))

## 3장. 데이터 품질 진단 — 정제 · 결측치 · 이상치
**할 일**: 무엇을 손봐야 하는지 **세어서 확인**합니다. 여기서는 **고치지 않습니다** — 처리 방법은 4장에서 정합니다.

전처리는 갈래마다 다루는 문제와 방법이 다릅니다. 먼저 이름부터 구분하세요.

| 전처리 갈래 | 무엇을 다루나 | 이 데이터의 대상 | 어디서 |
| --- | --- | --- | --- |
| **정제**(cleaning) | 중복 행 · 잘못된 자료형 | 완전 중복 행, 문자열로 들어온 `t_dat` | 중복은 **결합 후에만** 판단 가능 → 6장 |
| **결측치 처리** | 값이 **비어 있음** | `club_member_status` 결측 | 3-4 · 3-5 |
| **이상치 처리** | 값은 있지만 **범위·정의를 벗어남** | `age` 0·999, `price` 0 이하, `sales_channel_id` 0 | 3-1 ~ 3-3 |

> 세 갈래는 **처리 방법이 서로 다릅니다.** 결측치는 채우거나 지우고, 이상치는 지우거나 누르거나(clip) 그대로 두고 표시만 합니다. 그래서 "무엇이 문제인가"부터 이름을 붙여 구분하는 것입니다.

In [ ]:
# [제공 코드] 함정 1) 나이에 비현실적인 값이 있다
ages = sorted(cu['age'].dropna().unique())
print('가장 작은 값 5개:', ages[:5])
print('가장 큰 값 5개  :', ages[-5:])
print('10세 미만:', int((cu['age'] < 10).sum()), '명 / 100세 초과:',
      int((cu['age'] > 100).sum()), '명')

plt.figure(figsize=(7, 3))
sns.histplot(cu['age'], bins=40, color='#4C72B0')
plt.title('나이 분포 — 양쪽 끝에 비현실적인 값이 섞여 있다')
plt.xlabel('age'); plt.ylabel('고객 수')
plt.show()

print('-> 지울 것인가, 대체할 것인가? 연령대 파생을 하려면 이 결정이 먼저다.')

### 3-2. 이상치를 규칙으로 잡아보기 — IQR · Z-score (day08)
통계 규칙이 이상치를 **자동으로** 걸러 줄 것 같지만, 이 데이터에서는 그렇지 않습니다. 직접 계산해 확인하세요.

In [ ]:
# [제공 코드] IQR 1.5배 규칙과 Z-score 를 age 에 실제로 적용해 봅니다
q1, q3 = cu['age'].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('IQR 규칙 — 정상 범위:', round(lo, 1), '~', round(hi, 1))
print('IQR 규칙으로 잡히는 이상치:', int(((cu['age'] < lo) | (cu['age'] > hi)).sum()), '명')

z = (cu['age'] - cu['age'].mean()) / cu['age'].std()
print('Z-score(|z|>3) 로 잡히는 이상치:', int((z.abs() > 3).sum()), '명')

under10 = cu['age'] < 10
print()
print('10세 미만(921명) 중 IQR 규칙이 잡아내는 수:', int((under10 & (cu['age'] < lo)).sum()))
print('10세 미만(921명) 중 Z-score 가 잡아내는 수  :', int((under10 & (z.abs() > 3)).sum()))
print('-> 두 규칙 모두 0명! IQR 하한이 음수라 자동 통과하고, 999 같은 극단값이 표준편차를',
      '키워 Z-score 기준을 무디게 만들기 때문이다.')
z999 = z[cu['age'] == 999].iloc[0]
print('-> 반대로 나이 999는? IQR·Z-score 둘 다 확실히 잡는다(z =', round(z999, 1), ').')
print('-> 결론: 큰 이상치는 통계 규칙이 잡아주지만, "10세 미만은 쇼핑 고객일 수 없다" 같은',
      '도메인 판단은 규칙이 대신해 주지 못한다. 두 가지를 함께 써야 한다.')

In [ ]:
# [제공 코드] 함정 2) 0원 이하 거래가 있다
print('price <= 0 인 거래:', int((tr['price'] <= 0).sum()), '건')
print('price 요약:'); display(tr['price'].describe())
print('-> 무료 배포? 오류 입력? 환불? 해석이 여러 가지다. 빼든 남기든 이유를 적어야 한다.')

In [ ]:
# [제공 코드] 함정 3) 채널 코드에 정의되지 않은 값이 섞여 있다
print('채널 코드:', sorted(tr['sales_channel_id'].unique()))
print('(1 = 오프라인, 2 = 온라인 -- 그럼 0 은?)')
print('0 인 거래:', int((tr['sales_channel_id'] == 0).sum()), '건')

# 함정 4) 멤버십에 결측과 탈퇴 고객이 있다
print()
print('멤버십 결측:', int(cu['club_member_status'].isna().sum()), '명')
print('탈퇴(LEFT CLUB):', int((cu['club_member_status'] == 'LEFT CLUB').sum()), '명')
print('-> 분석 대상에 포함할지 정하고, 그 이유를 적는다.')

### 3-5. 결측치 2) 채우는 방법 세 가지 비교 — `fillna` vs `ffill`/`bfill`
결측을 채우는 방법은 하나가 아니고, **어느 것이 맞는지는 데이터의 성격이 결정합니다.** 직접 비교해 확인하세요.

In [ ]:
# [제공 코드] 결측 채우기 3가지 방법 비교 (복사본으로 시험 — cu 자체는 바꾸지 않음)
demo = cu['club_member_status'].copy()
print('원본 결측:', int(demo.isna().sum()), '명')

filled_const = demo.fillna('Unknown')   # 상수로 채우기
filled_ffill = demo.ffill()             # 바로 앞 값으로 채우기
filled_bfill = demo.bfill()             # 바로 뒤 값으로 채우기

print('상수(Unknown) 채움 후 결측:', int(filled_const.isna().sum()))
print('ffill 채움 후 결측     :', int(filled_ffill.isna().sum()))
print('bfill 채움 후 결측     :', int(filled_bfill.isna().sum()))
print()
print('-> 셋 다 기술적으로는 결측을 채운다. 하지만 cu 는 고객이 무작위 순서로 나열된 표라서',
      '"바로 앞/뒤 고객의 멤버십 상태"에는 아무 의미가 없다 -> 순서에 의미가 없는 속성',
      '데이터엔 상수 채우기(또는 4장에서 정할 규칙)가 맞고, ffill/bfill 은 시계열처럼',
      '순서 자체에 의미가 있을 때(예: 센서 로그) 쓰는 도구다.')

**데이터를 읽을 때 함께 알아둘 것**

| 함정 | 무엇을 조심하나 |
| --- | --- |
| **매출 계산** | `tr` 에 **수량(quantity) 컬럼이 없습니다.** `price` 합계는 매출이 아니라 **"구매 단가의 총합"** 입니다 |
| **통화·단위** | 원 통화 금액이 아니라 **정규화된 상대값**(0 ~ 0.422)입니다. 어떤 통화로도 환산하지 말고 **상대 비교에만** 쓰세요 |
| **시간** | `t_dat` 는 아직 문자열입니다. 월·요일 파생 전에 **`pd.to_datetime` 변환이 먼저**입니다. 기간은 2019년 1년치라 **연도 간 비교는 불가능**합니다 |
| **표본 편향** | H&M 전체가 아니라 **공개 표본**이고 **온라인 기록이 상대적으로 많습니다**(온라인 103,836건 vs 오프라인 45,861건). 채널 비중을 비교할 때 이 편향 때문에 **온라인이 커 보이는 것**일 수 있으니, 건수만으로 "온라인이 우세하다"고 단정하지 마세요 |
| **시즌성** | 특정 월의 급증이 **연말·행사 효과**일 수 있습니다. 한 달 치가 높다고 "성장"으로 단정하지 마세요 |

**여기까지 되면 통과**: 3-1 ~ 3-5 의 출력을 모두 확인했으면 됩니다(이 장은 채점하지 않습니다 — 눈으로 확인하는 것 자체가 목표).

### 3-6. 이상치의 두 층위 — 값 수준 vs 개체 수준
3장에서 본 것은 모두 **값 수준**(컬럼 하나의 값이 이상한가)입니다. 값은 하나하나 정상인데 **그 조합이나 합계가 이상한** 경우가 따로 있습니다.

| 층위 | 무엇을 보는가 | 이 데이터의 예 | 찾는 방법 |
| --- | --- | --- | --- |
| **값 수준** | 컬럼 하나의 값 | 나이 999, 가격 0, 채널 0 | 3-1 ~ 3-4 에서 한 것 |
| **개체 수준** | 한 고객 / 한 거래를 통째로 | 1년에 수백 건을 산 고객 | 고객별로 묶어 구매 건수·총 결제금액의 분포에서 극단값을 찾는다 |

개체 수준 이상치는 **사람이 아닐 수도 있습니다**(재판매업자·기업 구매·중복 기록). 그런 몇 명이 평균을 끌어올리면 "우리 고객은 1년에 평균 몇 건 산다" 가 통째로 틀립니다.

> 개체 수준 탐색은 고객별 집계가 필요하니 **02 에서** 합니다. 지금은 4장 결정 기록표에 **어떻게 다룰지 한 줄만** 정해 두세요 — "남겨 두고 02 에서 확인한다" 도 정당합니다. **정하지 않는 것만 문제입니다.**

## 4장. 전처리 규칙 수립 — 항목별로 어떻게 처리할지 정한다
**할 일**: 3장에서 센 것들을 **어떻게 처리할지 고르고 이유를 적습니다.** 코드는 6장에서 적용합니다. **"정답"은 없고, 이유 없는 선택만 문제입니다.**

### 4-1. 정제 — 완전 중복 행 · 자료형

| 항목 | 어떻게 처리하나 | 판단 |
| --- | --- | --- |
| 완전 중복 행 | `drop_duplicates()` 로 제거 / 그대로 유지 | 이 데이터엔 **수량 컬럼이 없어** 정상 재구매와 입력 오류를 구분할 수 없습니다. 제거가 무난하지만 근거를 적으세요 |
| `t_dat` 자료형 | `pd.to_datetime` 으로 날짜형 변환 | **선택이 아닙니다** — 월·요일 파생을 하려면 반드시 먼저 해야 합니다 |

> 완전 중복 행은 **결합 후에만** 판단할 수 있습니다. 거래 한 줄만으로는 진짜 중복인지 알 수 없고, 고객·상품 정보까지 붙은 완전한 행이라야 비교가 됩니다 → 6장.

### 4-2. 결측치 처리 — 멤버십(`club_member_status`)
결측 **2,500명**, `LEFT CLUB`(탈퇴) **40명**.

| 처리 방법 | 트레이드오프 |
| --- | --- |
| **별도 범주로 채우기** — `fillna('Unknown')` | 행을 지키고 "모른다"도 정보로 남김 / 결측의 원인은 여전히 모름 |
| **행 제거** | 확실한 값만 남음 / 표본이 줄고, 결측이 무작위가 아니면 편향됨 |
| 최빈값 등으로 대체 | 결측이 사라짐 / 관측되지 않은 값을 넣는 것이라 멤버십별 분석 근거가 약해짐 |

> **`LEFT CLUB` 포함 여부를 반드시 명시하세요.** 탈퇴 전 구매도 실제 있었던 거래이니 **매출을 볼 때는 포함**이 자연스럽고, "지금 회원은 어떻게 행동하나" 를 볼 때는 **제외**가 맞습니다. 목적에 따라 갈립니다.

### 4-3. 이상치 처리 — 나이 · 가격 · 채널 코드 · 개체 수준

**1) 나이(`age`)** — 0·3·7·130·999 가 섞여 있고, 3-2 에서 봤듯 **통계 규칙만으로는 10세 미만을 하나도 못 걸러냅니다.**

| 처리 방법 | 트레이드오프 |
| --- | --- |
| **제거** — 범위만 남기기(예: `between(10, 99)`) | 단순하고 설명하기 쉬움 / 경계값이 다소 자의적 |
| **누르기(clip)** — 상·하한으로 압축 | 행 손실 없음 / 없던 값이 실제처럼 보임 |
| **결측 처리 후 대체** — 중앙값 등 | 표본 유지 + 극단값 영향 제거 / 관측 안 된 값이라 근거 약함 |
| **유지** | 정보 손실 없음 / 평균·표준편차가 크게 흔들림 |

> **`age_group` 파생은 이 결정 다음입니다.** 순서를 뒤집으면 999세가 90대 그룹으로 섞입니다.

**2) 가격(`price`) 0 이하** — **1,004건**. 무료 배포인지 오류인지 환불인지 **데이터만으로는 알 수 없습니다.**

| 처리 방법 | 트레이드오프 |
| --- | --- |
| **제거** | 집계가 단순해짐 / 사은품 같은 정상 거래까지 지울 수 있음 |
| **플래그 + 유지** — `is_free` 참/거짓 컬럼 | 포함·제외 결과를 **둘 다** 볼 수 있음(실무에서 자주) / 컬럼이 늘고 집계마다 판단이 필요 |
| 대체(평균·중앙값) | 행 보존 / 0원이 정말 결측인지 알 수 없어 부정확 |
| 유지 | 정보 손실 없음 / 평균 단가·합계가 낮게 왜곡됨 |

**3) 채널 코드 0 (803건)** — 정의(1·2)에 없는 값이라 **분석 축으로 쓸 수 없습니다.** 제거하거나 별도 범주로 묶되, 803건을 버리는 것임을 적어 두세요.

**4) 개체 수준 이상치** — 3-6 참고. 지금은 "어떻게 다룰지" 한 줄만 정하고 탐색은 02 에서.

### 4-4. 처리 순서
**정제 → 결측치 → 이상치 → 파생** 순서를 지키세요. 이유는 두 가지입니다.

- **파생보다 이상치 제거가 먼저**: `age_group` 을 먼저 만들면 999세도 엉뚱한 그룹으로 섞여 이후 집계가 왜곡됩니다.
- **완전 중복 제거는 결합 이후**: 거래 한 줄만으로는 진짜 중복 구매인지 알 수 없습니다.

나이·멤버십처럼 **한 테이블에만 있는 컬럼**은 결합 전에 걸러도 후에 걸러도 결과가 같습니다.

### 4-5. 결정 기록표 (직접 채우세요)
**요구사항**: **내 선택·이유·잃은 것**을 모두 채우세요. 규칙이 무엇이든 상관없고 **이유가 없으면 통과가 아닙니다.**

| 갈래 | 항목 | 내 선택 | 이유 | 잃은 것 |
| --- | --- | --- | --- | --- |
| 정제 | 완전 중복 행 | 제거 | 수량 컬럼이 없어 정상 재구매와 입력 오류를 구분할 수 없음 | 혹시 있었을 정상 재구매 정보 |
| 결측치 | 멤버십 결측·LEFT CLUB | 결측→`Unknown`, `LEFT CLUB` 포함 | 결측도 정보로 보존, 탈퇴 전 구매도 실제 매출 | 결측의 진짜 원인은 알 수 없음 |
| 이상치 | 나이 | 10~99세만 남김 | 통계 규칙으로는 10세 미만이 안 걸러져 도메인 기준을 직접 정함 | 실제 10세 미만·100세 이상 고객이 있었다면 그 정보 |
| 이상치 | 가격 0 이하 | 제거 | 사은품인지 오류인지 구분 불가, 집계를 일관되게 | 사은품이었을 수 있는 정상 거래 |
| 이상치 | 채널 코드 0 | 제거 | 1·2 로 정의되지 않아 분석 축으로 쓸 수 없음 | 거래 803건 |
| 이상치 | 개체 수준(고객·거래) | 남겨 두고 02 에서 상위 고객 확인 | 지금 지우면 정상 우량 고객과 구분할 근거가 없음 | 그때까지 평균·합계에 영향이 섞임 |

**여기까지 되면 통과**: 여섯 항목 모두 **선택과 이유**가 채워져 있으면 됩니다. 코드 적용은 6장에서 합니다.

## 5장. 테이블 결합
**이 절에서 할 일**: `customer_id`·`article_id` 로 세 테이블을 하나로 합쳐 분석용 데이터셋을 만듭니다. 막히면 가이드의 "두 테이블 합치고 행 검증" 항목을 참고하세요.

**`inner` vs `left`** — `how='inner'` 는 **양쪽에 모두 키가 있는 행만** 남기고, `how='left'` 는 **왼쪽을 전부 남기고** 오른쪽 정보가 없으면 결측으로 채웁니다. 이 프로젝트는 고객·상품 정보가 **반드시 있어야** 이후 분석(연령대·상품군 비교 등)이 가능하므로 **`inner`** 로 결합합니다(고객·상품 정보가 없는 거래는 애초에 분석 대상이 아니라고 보는 선택입니다).

> **그럼 `left` 는 언제 필요한가** — 질문이 바뀌면 조인도 바뀝니다. 대표적인 경우가 "**가입은 했지만 한 번도 구매하지 않은 고객**은 몇 명인가?" 입니다. 이건 고객 테이블을 **왼쪽에 두고** 거래를 `left` 로 붙여야 보입니다 — 구매 기록이 없는 고객이 결측 행으로 남기 때문입니다. `inner` 로는 그 고객들이 조용히 사라져 애초에 없었던 것처럼 됩니다.
> 다만 **이 데이터에서 그 답은 0명**입니다(고객 테이블의 95,516명이 모두 거래에 등장합니다). "발제문에 나온 주의사항이 내 데이터에도 해당되는지"는 **직접 확인해야 알 수 있습니다** — 이것도 여러분이 검증해 볼 만한 질문입니다.
> 대신 이 데이터에는 **반대 방향의 손실**이 있습니다. `inner` 로 결합하면 거래 150,500건 중 **33,527건(22.3%)이 사라집니다** — 그 거래를 한 고객이 고객 테이블에 없기 때문입니다(거래에는 123,053명이 등장하는데 고객 테이블에는 95,516명뿐입니다). 즉 이 프로젝트가 `inner` 를 고르는 순간 **"고객 정보를 아는 거래만 분석한다"** 고 선언하는 것이고, 그래서 여기서 계산할 매출은 **전체 매출이 아닙니다.** 전체 거래 금액을 보고 싶다면 거래를 왼쪽에 두고 `left` 로 붙여 나이·멤버십을 결측으로 남기는 편이 맞습니다.
> 이 노트북의 정제본은 "고객·상품 정보가 붙은 구매 1건"을 관측단위로 하므로 `inner` 로 만듭니다(자가채점도 그 기준입니다). **조인 방식은 질문이 정한다** — 이것이 이 절의 핵심이고, 무엇을 골랐든 **몇 건이 왜 빠졌는지 적어 두는 것**이 요구사항입니다.

> **모든 테이블을 반드시 결합할 필요는 없습니다.** 분석 목적에 따라 조인 범위를 줄이는 것이 오히려 낫습니다 — 예를 들어 "연령대별 구매 패턴" 만 볼 거라면 거래 + 고객 두 개로 충분하고, 상품 25컬럼을 붙이면 메모리만 쓰고 얻는 게 없습니다. 이 노트북은 02·03 이 상품군 분석까지 하므로 세 개를 다 붙입니다.

**결합 후 행 증가 여부 검증** — merge 는 키가 중복되면 행이 **늘어날 수도** 있습니다. `cu`·`ar` 의 키(`customer_id`·`article_id`)가 각각 유니크한지 미리 확인하면, 결합 후 행수가 예상 밖으로 튀지 않는다는 것을 보장할 수 있습니다.

**중복 컬럼 확인** — 두 테이블을 합칠 때 이름이 겹치는 컬럼이 있으면 pandas 가 `_x`/`_y` 접미사를 붙입니다. 미리 `set(a.columns) & set(b.columns)` 로 확인하는 습관을 들이면 좋습니다.

아래는 **제공 코드**입니다 — 실행해서 결과를 확인하세요.

In [ ]:
# [제공 코드] 결합 전 검증 — 키 유니크 여부 + 컬럼 겹침 확인
print('cu 의 customer_id 유니크?', cu['customer_id'].is_unique)
print('ar 의 article_id 유니크?', ar['article_id'].is_unique)
print('tr & cu 공통 컬럼:', set(tr.columns) & set(cu.columns))
print('tr & ar 공통 컬럼:', set(tr.columns) & set(ar.columns))
print('cu & ar 공통 컬럼:', set(cu.columns) & set(ar.columns))

In [ ]:
# [제공 코드] 거래를 기준으로 고객·상품 속성을 inner join 으로 결합합니다.
before_rows = len(tr)
df = tr.merge(cu, on='customer_id', how='inner').merge(ar, on='article_id', how='inner')
print('행 증가 여부 검증: 거래', before_rows, '->', '병합 결과', len(df))
print('병합 결과:', df.shape)
display(df.head(3))

In [ ]:
# [자가채점]
assert df.shape == (116973, 34)
print("✅ 조인 직후 행수 확인 통과!")

**여기까지 되면 통과**: `df` 가 만들어지고 위 자가채점이 통과하면 됩니다.

## 6장. 전처리 규칙 적용 & 파생변수

**6-1. 전처리 실행** — 4장에서 정한 규칙을 `df` 에 적용하세요. 순서는 4-4 대로 **정제(완전 중복) → 결측치(멤버십) → 이상치(나이·가격·채널)** 입니다.

> **처리 로그를 남기세요** — 단계별 전후 행수를 `log` 리스트에 쌓아 마지막에 표로 출력합니다. "몇 건이 왜 빠졌는지"를 설명할 수 있어야 합니다.

> 4장에서 기준선(완전 중복 제거·나이 10~99·가격>0·채널 1·2·멤버십→Unknown)과 **다른 규칙**을 골랐다면 행수가 정답 예시본과 달라지는 것이 **정상**입니다.

> 💡 나이 범위는 `df['age'].between(10, 99)` 가 `(df['age'] >= 10) & (df['age'] <= 99)` 와 같은 결과를 내면서 더 읽기 쉽습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건에 맞는 행만 남기는 필터를 순서대로 적용하고, 결측은 fillna 로 채운다.
- 매 단계 전후 len(df) 를 log 리스트(딕셔너리들)에 쌓고, 마지막에 pd.DataFrame(log) 로 표를 만들어 확인한다.

세부구현(예시 하나 — 그대로 안 써도 됩니다):
1. log = [] 로 시작한다
2. before = len(df) 를 찍고 df = df.drop_duplicates() 로 완전 중복 행을 제거한 뒤 log.append({...}) 로 기록한다
3. 같은 방식으로 age 범위 필터, price>0 필터, sales_channel_id isin 필터를 적용하고 각각 log 에 기록한다
4. club_member_status 의 결측을 정한 규칙대로 처리하고 log 에 기록한다
5. display(pd.DataFrame(log)) 로 처리 로그 표를 출력한다
```

</details>

In [ ]:
log = []

before = len(df)
df = df.drop_duplicates()
log.append({'단계': '완전 중복 제거', '처리전': before, '처리후': len(df),
            '제거': before - len(df)})

before = len(df)
df = df[df['age'].between(10, 99)]   # (age>=10)&(age<=99) 와 동일 - 더 읽기 쉽다
log.append({'단계': '나이 이상치 제거', '처리전': before, '처리후': len(df),
            '제거': before - len(df)})

before = len(df)
df = df[df['price'] > 0]
log.append({'단계': '가격 0 이하 제거', '처리전': before, '처리후': len(df),
            '제거': before - len(df)})

before = len(df)
df = df[df['sales_channel_id'].isin([1, 2])]
log.append({'단계': '채널 오류코드 제거', '처리전': before, '처리후': len(df),
            '제거': before - len(df)})

before = len(df)
df['club_member_status'] = df['club_member_status'].fillna('Unknown')
log.append({'단계': '멤버십 결측 -> Unknown', '처리전': before, '처리후': len(df),
            '제거': 0})

print('처리 로그:')
display(pd.DataFrame(log))
print('전처리 규칙 적용 후:', df.shape)

**6-2. 파생변수 만들기** — **왜**: 02·03 이 연령대·시간대별 분석을 바로 할 수 있도록, 정제본에 미리 파생 컬럼을 심어 둡니다.

> 🔵 **스키마 계약 — 이 세 컬럼은 이름을 반드시 지켜야 합니다** (02·03 이 이 이름을 그대로 씁니다):
1. **`month`**(정수 1~12) — `t_dat` 를 `pd.to_datetime` 으로 날짜형으로 바꾼 뒤, `.dt.month`.
2. **`weekday`**(요일 **이름**, 예: `Monday`) — 같은 날짜형에서 `.dt.day_name()`.
3. **`age_group`**(정수, 10·20·…·90) — `age` 를 10살 단위로 묶습니다(`age // 10 * 10`, 정수형).

막히면 가이드의 "문자열 -> 날짜형"·"파생 컬럼 만들기" 항목을 참고하세요.

<details><summary>힌트</summary>

```text
세부구현:
1. t_dat 를 to_datetime 으로 날짜형으로 변환해 다시 담는다
2. dt.month 로 month 컬럼을, dt.day_name() 으로 weekday 컬럼을 만든다
3. age 를 10 으로 정수 나눗셈한 뒤 10 을 곱하고 정수형으로 바꿔 age_group 을 만든다
```

</details>

In [ ]:
df['t_dat'] = pd.to_datetime(df['t_dat'])
df['month'] = df['t_dat'].dt.month
df['weekday'] = df['t_dat'].dt.day_name()
df['age_group'] = (df['age'] // 10 * 10).astype(int)
print('파생변수 추가 후:', df.shape)
display(df[['t_dat', 'month', 'weekday', 'age', 'age_group']].head())

**여기까지 되면 통과**: 처리 로그 표가 출력되고, `month`·`weekday`·`age_group` 세 컬럼이 만들어져 있으면 됩니다(행수는 규칙에 따라 달라도 정상입니다).

## 7장. 정제본 저장
**이 절에서 할 일**: 지금까지 정리한 `df` 를 CSV 로 저장합니다.

> 이 파일이 **02·03 의 입력**입니다. 여기까지 끝내야 다음 노트북이 돌아갑니다.
> (정답 예시본 02·03 은 파일이 없으면 표준 규칙으로 재현해 실행되지만, **여러분이 정한 규칙은 반영되지 않습니다** — 순서대로 01 을 먼저 끝내세요.)

- **저장 경로**: `output/hm_clean.csv` (인덱스 없이, `index=False`).
- 아래는 **제공 코드**입니다 — 저장 직전 `df` 의 shape·컬럼 목록을 출력하고, 저장한 뒤 다시 읽어 행수를 확인합니다.

> 🔒 이 셀은 **`month`·`weekday`·`age_group` 세 파생 컬럼이 모두 있을 때만 저장**합니다. 6장을 건너뛴 채 저장하면 정제되지 않은 데이터가 02·03 으로 흘러가 엉뚱한 분석 결과가 나오기 때문입니다. 저장이 안 됐다는 메시지가 보이면 6장으로 돌아가세요.
> 필터 규칙(나이·가격·채널·멤버십)은 **여러분의 선택**이므로 저장을 막을 조건이 아닙니다 — 남아 있는 항목은 알려만 주면 됩니다.

> 참고로 계약 기준선(완전 중복 제거 539행·나이 10~99·가격>0·채널 1·2·멤버십→Unknown)을 그대로 적용하면 최종 **113,218행 x 37열**(원본 조인 34열 + 파생 3열)이 됩니다. 여러분이 다른 규칙을 골랐다면 행수가 이 값과 달라지는 것이 정상입니다(이 저장 단계는 채점하지 않습니다) — 다만 **`month`·`weekday`·`age_group` 세 컬럼 이름은 그대로 유지**하세요.

In [ ]:
# [제공 코드] 정제본을 저장합니다.
# 저장 전에 '6장을 끝냈는가' 를 확인합니다 — 파생 컬럼 세 개가 저장 조건입니다.
# (필터 규칙은 여러분의 선택이라 조건에 넣지 않고 아래에서 확인만 합니다.)
print('저장 직전 최종 df')
print('shape:', df.shape)
print('컬럼:', list(df.columns))
print()

need = ['month', 'weekday', 'age_group']
missing = [c for c in need if c not in df.columns]

if missing:
    print('아직 저장하지 않았습니다 — 6장이 끝나지 않았습니다.')
    print('  없는 파생 컬럼:', ', '.join(missing))
    print('  6장을 마친 뒤 이 셀을 다시 실행하세요.')
    print('  (이 세 컬럼 이름은 02·03 이 직접 쓰므로 반드시 이 이름이어야 합니다.)')
else:
    # 남아 있는 함정이 있으면 알려만 줍니다 — 일부러 남긴 것이라면 그대로 진행해도 됩니다.
    # (컬럼을 지우기로 했다면 그 검사는 건너뜁니다 — 그것도 여러분의 선택입니다.)
    left = []
    if 'age' in df.columns and ((df['age'] < 10) | (df['age'] > 99)).any():
        left.append('10세 미만 또는 99세 초과 나이')
    if 'price' in df.columns and (df['price'] <= 0).any():
        left.append('0 이하 가격')
    if 'sales_channel_id' in df.columns and (~df['sales_channel_id'].isin([1, 2])).any():
        left.append('1·2 가 아닌 채널 코드')
    if 'club_member_status' in df.columns and df['club_member_status'].isna().any():
        left.append('멤버십 결측')
    if left:
        print('참고 — 아래 항목이 아직 남아 있습니다. 의도한 선택이면 그대로 두세요.')
        for item in left:
            print('  -', item)
        print('  (남겨 두면 02·03 의 일부 예시가 그 값을 자동으로 걸러내고 알려 줍니다.)')
        print()

    os.makedirs('output', exist_ok=True)
    clean_path = 'output/hm_clean.csv'
    df.to_csv(clean_path, index=False)
    print('저장 완료:', clean_path)

    reloaded = pd.read_csv(clean_path)
    print('재로딩 확인 shape:', reloaded.shape)
    print('재로딩 확인 컬럼:', list(reloaded.columns))

**다음 단계** — 정제본을 저장했다면 `02_데이터_EDA.ipynb` 로 넘어가세요. 그 노트북이 방금 저장한 `output/hm_clean.csv` 를 읽어 EDA·시각화를 진행합니다.